In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, to_hetero
from torch_geometric.loader import DataLoader
from halide_gnn_cost_model.data import PipelineDataset
from pathlib import Path

In [ ]:
# Load dataset
dataset = PipelineDataset(Path("../resources/pipelines"))
data = dataset[0]  # Get the first pipeline graph
print(data.metadata)

In [ ]:
data

In [ ]:
class PipeGAT(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, num_layers=2, heads=1):
        super(PipeGAT, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.convs.append(GATConv(-1, hidden_channels, heads=heads, concat=True))
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(-1, hidden_channels, heads=heads, concat=True))
        self.convs.append(GATConv(-1, out_channels, heads=heads, concat=False))

    def forward(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=0.5, training=self.training)
        x = self.convs[-1](x, edge_index)
        return x

In [ ]:
gat = PipeGAT(hidden_channels=32, out_channels=32, num_layers=3, heads=1)
gat = to_hetero(gat, data.metadata(), aggr="sum")

In [ ]:
out = gat(data.x_dict, data.edge_index_dict)
out["function"][0]

In [ ]:
class PipelineModel(torch.nn.Module):
    def __init__(self, gnn, out_channels, num_runtime):
        super(PipelineModel, self).__init__()
        self.function_gnn = gnn
        self.pipeline_lin = torch.nn.Linear(out_channels, num_runtime)

    def forward(self, data, ptr=None):
        x_dict = data.x_dict
        edge_index_dict = data.edge_index_dict
        out = self.function_gnn(x_dict, edge_index_dict)
        # Get the feature of the pipeline node
        idx = 0 if ptr is None else ptr
        pipeline_feat = out["function"][idx]
        # Predict the runtime
        x = self.pipeline_lin(pipeline_feat)
        run_time = torch.exp(x)
        return run_time

In [ ]:
model = PipelineModel(gat, 32, 5)
model

In [ ]:
data_loader = DataLoader(dataset, batch_size=4, shuffle=True)
data_loader

In [ ]:
# Train loop (example)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = torch.nn.L1Loss()

for epoch in range(100):
    model.train()
    total_loss = 0
    for batch in data_loader:
        optimizer.zero_grad()
        pred = model(batch, batch["function"].ptr[:-1])
        # Assuming batch.y contains the true runtimes
        loss = criterion(pred.reshape(-1), batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(data_loader)}")

In [ ]:
torch.set_printoptions(precision=4)
print(model(data))